# OpsSim-AI: Before vs After LoRA Comparison

This notebook compares a **base model** (before training) against the **base + trained LoRA adapter** (after GRPO training) on cascading DevOps incident scenarios.

**Evaluation integrity guarantees:**
- Greedy decoding (temperature=0) with fixed torch seed for full determinism
- Adapter state explicitly verified before each generation pass
- KV cache cleared between base and adapter runs to prevent state leakage
- Generous token budget to prevent output truncation corrupting JSON metrics
- Rich metrics including partial correctness, optimal-path membership, and key completeness

## 1. Configuration

Set your base model, adapter repo, and HF token below.

In [ ]:
# ====== CONFIGURE THESE ======
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_REPO = "meancodi/opssim-qwen-3b-v1"
HF_TOKEN = ""  # paste your token here (required for gated models / private adapters)

# Inference settings — identical for both base and adapter runs
MAX_NEW_TOKENS = 512   # generous to avoid truncation corrupting JSON
TEMPERATURE = 0.0      # greedy decoding for deterministic comparison
TOP_P = 1.0            # no nucleus sampling (greedy overrides this)
PRECISION = "bf16"     # bf16 | fp16 | fp32
SEED = 42              # fixed seed for torch RNG (deterministic generation)

# How many scenario+step pairs to test (more = more statistically reliable)
NUM_EXAMPLES = 15      # covers diverse scenarios; use 5 for a quick test

## 2. Install Dependencies & Clone Repo

In [ ]:
!pip install -q torch transformers accelerate peft huggingface_hub
!pip install -q jmespath pydantic

import os

# Authenticate with HF Hub (needed for gated models and private repos)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Authenticated with Hugging Face Hub.")
else:
    print("WARNING: No HF_TOKEN set. Gated models / private adapters will fail to load.")

# Clone the repo to get tasks/cascade.json and helper modules
REPO_DIR = "OpsSim-AI"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/nithishgouds/Meta-x-OpenEnv-x-Pytorch-Hackathon.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 3. Import Repo Utilities

Reuse the exact functions from the training pipeline — no rewriting.

In [ ]:
import copy
import json
from typing import Any
from collections import OrderedDict

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Reuse directly from the repo ---
from generate_sft_dataset import (
    load_scenarios,
    apply_effects,
    build_prompt,
    build_assistant,
    evaluate_condition,
    detect_anomalies,
    find_agent_for_action,
    classify_unsafe_actions,
    AGENT_DOMAIN_MAP,
)

from run_trained_inference import (
    extract_json,
    replay_state,
    render_chat,
)

# Environment for episode rollouts (same one used during GRPO training)
from env import DevOpsEnv
from models import OpsSIMAction

# Domain agent lookup (same logic as train_grpo.py)
DOMAIN_TO_AGENT = {domain: agent for agent, domain in AGENT_DOMAIN_MAP.items()}
EXECUTION_AGENTS = list(AGENT_DOMAIN_MAP.keys())

def domain_agent_for_action(action: str, scenario: dict) -> str:
    """Find the correct domain agent for an action (replicates train_grpo._domain_agent_for_action)."""
    action_domains = scenario.get("action_domains", {}) or {}
    for domain, actions in action_domains.items():
        if action in actions:
            return DOMAIN_TO_AGENT.get(domain, "AppOps")
    return "AppOps"

print("All repo utilities imported successfully (including DevOpsEnv for episode rollouts).")

## 4. Load Scenarios

In [ ]:
scenarios = load_scenarios("tasks/cascade.json")
print(f"Loaded {len(scenarios)} scenarios:")
for s in scenarios:
    optimal = s.get("optimal_solution_path", [])
    print(f"  {s['scenario_id']}  ({len(optimal)} steps)")

## 5. Build Test Examples

Pick diverse scenario+step combinations across different scenarios and step indices.

In [ ]:
import random
random.seed(SEED)

test_examples = []
candidates = []
for scenario in scenarios:
    optimal = scenario.get("optimal_solution_path", []) or []
    for step_idx in range(1, len(optimal) + 1):
        candidates.append((scenario, step_idx))

selected = random.sample(candidates, min(NUM_EXAMPLES, len(candidates)))

for scenario, step_idx in selected:
    optimal = scenario.get("optimal_solution_path", []) or []
    total_steps = len(optimal)
    completed = optimal[:step_idx - 1]
    state, rewards = replay_state(scenario, completed)
    gold_action = optimal[step_idx - 1]
    gold_agent, gold_domain = find_agent_for_action(
        gold_action, scenario.get("action_domains", {})
    )

    prompt = build_prompt(
        scenario=scenario,
        state=state,
        step_idx=step_idx,
        total_steps=total_steps,
        completed=completed,
        completed_rewards=rewards,
        candidate=None,
    )

    gold_response = build_assistant(
        scenario=scenario,
        state=state,
        step_idx=step_idx,
        total_steps=total_steps,
        completed=completed,
        completed_rewards=rewards,
        action=gold_action,
    )

    unsafe_actions = [
        u["action"] for u in classify_unsafe_actions(scenario, state, set(completed), gold_action)
    ]

    # Penalized actions (explicit penalty in scenario definition)
    penalized_actions = list((scenario.get("penalties", {}) or {}).keys())

    test_examples.append({
        "scenario_id": scenario["scenario_id"],
        "step_idx": step_idx,
        "total_steps": total_steps,
        "prompt": prompt,
        "gold_action": gold_action,
        "gold_agent": gold_agent,
        "gold_response": gold_response,
        "unsafe_actions": unsafe_actions,
        "penalized_actions": penalized_actions,
        "optimal_path": optimal,  # full path for partial credit
    })

# Summary
unique_scenarios = len(set(ex["scenario_id"] for ex in test_examples))
print(f"Built {len(test_examples)} test examples from {unique_scenarios} unique scenarios:")
for ex in test_examples:
    print(f"  {ex['scenario_id']}  step {ex['step_idx']}/{ex['total_steps']}  gold={ex['gold_action']}")

## 6. Load Models

We load **two fully independent model instances** to eliminate any shared-state or KV cache
contamination between the "before" and "after" runs. Each model gets its own weights in memory.

In [ ]:
torch_dtype = {
    "bf16": torch.bfloat16,
    "fp16": torch.float16,
    "fp32": torch.float32,
}[PRECISION]

hf_auth = {"token": HF_TOKEN} if HF_TOKEN else {}

print(f"Loading tokenizer from {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, **hf_auth)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- Base model (before training) — standalone instance ---
print(f"Loading BASE model ({PRECISION})...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch_dtype,
    device_map="auto",
    trust_remote_code=True,
    **hf_auth,
)
base_model.eval()
base_model.config.use_cache = True
print(f"  Base model params: {sum(p.numel() for p in base_model.parameters()):,}")

# --- Adapter model (after training) — separate instance with LoRA merged ---
print(f"\nLoading ADAPTER model from {ADAPTER_REPO}...")
adapter_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch_dtype,
    device_map="auto",
    trust_remote_code=True,
    **hf_auth,
)
adapter_model = PeftModel.from_pretrained(adapter_base, ADAPTER_REPO, **hf_auth)
adapter_model.eval()
adapter_model.config.use_cache = True

# Verify the adapter is active
trainable = sum(p.numel() for p in adapter_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in adapter_model.parameters())
print(f"  Adapter model params: {total:,} (trainable LoRA: {trainable:,})")

# Sanity check: adapter layers exist and are enabled
active_adapters = adapter_model.active_adapters
print(f"  Active adapters: {active_adapters}")
assert active_adapters, "ERROR: No active adapters found — adapter not loaded correctly!"

print("\nBoth models loaded as independent instances. No shared state.")

## 7. Inference Helper

Deterministic generation with explicit seed reset and cache clearing before each call.
Generation parameters are passed explicitly (not from globals) to guarantee identical settings.

In [ ]:
def generate(model, prompt: str, max_new_tokens: int = MAX_NEW_TOKENS,
             temperature: float = TEMPERATURE, top_p: float = TOP_P,
             seed: int = SEED) -> tuple[str, bool]:
    """Generate a response. Returns (text, was_truncated).

    Deterministic: resets torch seed before each call and clears KV cache.
    """
    # Reset RNG for deterministic generation
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    rendered = render_chat(tokenizer, prompt)
    inputs = tokenizer(rendered, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy — fully deterministic (temperature=0 equivalent)
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # Detect truncation: if we hit max_new_tokens and didn't end with EOS
    was_truncated = (
        len(generated_ids) >= max_new_tokens
        and (len(generated_ids) == 0 or generated_ids[-1].item() != tokenizer.eos_token_id)
    )

    return text, was_truncated

# Quick sanity: verify determinism with a trivial prompt
torch.manual_seed(SEED)
print(f"Inference helper ready. Seed={SEED}, max_new_tokens={MAX_NEW_TOKENS}, greedy decoding.")

## 8. Metrics

Rich evaluation covering:
- **Structural**: valid JSON, all 6 required keys present, key completeness ratio
- **Correctness**: exact action match, exact agent match, action on optimal path
- **Safety**: unsafe action chosen, action is penalized
- **Quality**: output truncation detected, confidence value present and valid

In [ ]:
REQUIRED_KEYS = {"analysis", "plan", "next_action", "target_agent", "reasoning", "confidence"}

def evaluate_response(raw_output: str, was_truncated: bool, example: dict) -> dict:
    """Score a single model response with rich metrics."""
    parsed = extract_json(raw_output)

    metrics = {
        # Structural
        "valid_json": parsed is not None,
        "has_all_keys": False,
        "key_completeness": 0.0,     # fraction of 6 required keys present
        "was_truncated": was_truncated,
        # Correctness
        "correct_action": False,
        "correct_agent": False,
        "action_on_optimal_path": False,  # partial credit: wrong step but right scenario action
        # Safety
        "is_unsafe": False,
        "is_penalized": False,
        # Quality
        "has_valid_confidence": False,
        "has_plan": False,
        # Raw predictions
        "predicted_action": None,
        "predicted_agent": None,
    }

    if parsed is None:
        return metrics

    # Key completeness
    present_keys = REQUIRED_KEYS.intersection(parsed.keys())
    metrics["key_completeness"] = len(present_keys) / len(REQUIRED_KEYS)
    metrics["has_all_keys"] = len(present_keys) == len(REQUIRED_KEYS)

    predicted_action = parsed.get("next_action", "")
    predicted_agent = parsed.get("target_agent", "")
    metrics["predicted_action"] = predicted_action
    metrics["predicted_agent"] = predicted_agent

    # Exact correctness
    metrics["correct_action"] = predicted_action == example["gold_action"]
    metrics["correct_agent"] = predicted_agent == example["gold_agent"]

    # Partial correctness: is the predicted action at least on the optimal path?
    optimal_path = example.get("optimal_path", [])
    metrics["action_on_optimal_path"] = predicted_action in optimal_path

    # Safety
    metrics["is_unsafe"] = predicted_action in example["unsafe_actions"]
    metrics["is_penalized"] = predicted_action in example.get("penalized_actions", [])

    # Quality signals
    confidence = parsed.get("confidence")
    metrics["has_valid_confidence"] = (
        confidence is not None
        and isinstance(confidence, (int, float))
        and 0.0 <= float(confidence) <= 1.0
    )
    plan = parsed.get("plan")
    metrics["has_plan"] = isinstance(plan, list) and len(plan) > 0

    return metrics

print("Rich metrics helper ready. Tracking 13 evaluation dimensions.")

## 9. Run Before vs After Comparison

In [ ]:
results = []

for i, example in enumerate(test_examples):
    print(f"\n{'='*80}")
    print(f"Example {i+1}/{len(test_examples)}")
    print(f"Scenario: {example['scenario_id']}")
    print(f"Step: {example['step_idx']}/{example['total_steps']}")
    print(f"Gold action: {example['gold_action']}  (agent: {example['gold_agent']})")
    print(f"{'='*80}")

    # --- Before (standalone base model) ---
    print("\n[BEFORE] Generating with base model...")
    before_raw, before_truncated = generate(base_model, example["prompt"], seed=SEED)
    before_metrics = evaluate_response(before_raw, before_truncated, example)

    # --- After (separate adapter model instance) ---
    print("[AFTER]  Generating with base + adapter...")
    # Verify adapter is active before generation
    assert adapter_model.active_adapters, "Adapter not active!"
    after_raw, after_truncated = generate(adapter_model, example["prompt"], seed=SEED)
    after_metrics = evaluate_response(after_raw, after_truncated, example)

    results.append({
        "example": example,
        "before_raw": before_raw,
        "after_raw": after_raw,
        "before_metrics": before_metrics,
        "after_metrics": after_metrics,
    })

    # --- Side-by-side output ---
    print(f"\n--- BEFORE (base only) ---")
    print(before_raw[:600])
    if before_truncated:
        print("  ⚠ OUTPUT WAS TRUNCATED")
    print(f"\n--- AFTER (base + adapter) ---")
    print(after_raw[:600])
    if after_truncated:
        print("  ⚠ OUTPUT WAS TRUNCATED")

    print(f"\n--- Metrics ---")
    header_metrics = ["valid_json", "has_all_keys", "correct_action", "correct_agent",
                      "action_on_optimal_path", "is_unsafe", "was_truncated"]
    print(f"{'Metric':<25} {'Before':<12} {'After':<12}")
    print(f"{'-'*50}")
    for key in header_metrics:
        bval = str(before_metrics[key])
        aval = str(after_metrics[key])
        marker = ""
        if key in ("is_unsafe", "was_truncated"):
            marker = " !!" if after_metrics[key] else ""
        else:
            marker = " ✓" if after_metrics[key] and not before_metrics[key] else ""
        print(f"{key:<25} {bval:<12} {aval:<12}{marker}")
    print(f"{'predicted_action':<25} {str(before_metrics['predicted_action'] or '-'):<12} {str(after_metrics['predicted_action'] or '-'):<12}")
    print(f"{'predicted_agent':<25} {str(before_metrics['predicted_agent'] or '-'):<12} {str(after_metrics['predicted_agent'] or '-'):<12}")
    print(f"{'key_completeness':<25} {before_metrics['key_completeness']:.0%}{'':>9} {after_metrics['key_completeness']:.0%}")

print(f"\n\nDone! Ran {len(results)} comparisons.")

## 10. Aggregate Summary

In [ ]:
import math

n = len(results)
if n == 0:
    print("No results to summarize.")
else:
    # Compute all metrics
    all_metrics = [
        "valid_json", "has_all_keys", "correct_action", "correct_agent",
        "action_on_optimal_path", "is_unsafe", "is_penalized", "was_truncated",
        "has_valid_confidence", "has_plan",
    ]
    avg_metrics = ["key_completeness"]

    summary = {"before": {}, "after": {}}
    for phase in ["before", "after"]:
        key = f"{phase}_metrics"
        for m in all_metrics:
            summary[phase][m] = sum(1 for r in results if r[key][m]) / n
        for m in avg_metrics:
            summary[phase][m] = sum(r[key][m] for r in results) / n

    # Wilson score confidence interval for proportions
    def wilson_ci(p, n, z=1.96):
        if n == 0:
            return (0.0, 0.0)
        denom = 1 + z**2 / n
        center = (p + z**2 / (2 * n)) / denom
        spread = z * math.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
        return (max(0, center - spread), min(1, center + spread))

    print(f"\n{'='*75}")
    print(f"  AGGREGATE SUMMARY  ({n} examples, {len(set(r['example']['scenario_id'] for r in results))} scenarios)")
    print(f"{'='*75}")
    print(f"{'Metric':<25} {'Before':<12} {'After':<12} {'Delta':<10} {'95% CI (After)'}")
    print(f"{'-'*75}")

    display_order = [
        ("valid_json", "higher"),
        ("has_all_keys", "higher"),
        ("key_completeness", "higher"),
        ("correct_action", "higher"),
        ("correct_agent", "higher"),
        ("action_on_optimal_path", "higher"),
        ("has_valid_confidence", "higher"),
        ("has_plan", "higher"),
        ("is_unsafe", "lower"),
        ("is_penalized", "lower"),
        ("was_truncated", "lower"),
    ]

    for metric, direction in display_order:
        b = summary["before"][metric]
        a = summary["after"][metric]
        delta = a - b
        ci_lo, ci_hi = wilson_ci(a, n)

        if direction == "lower":
            arrow = "↓ ✓" if delta < 0 else ("↑ !!" if delta > 0 else "  →")
        else:
            arrow = "↑ ✓" if delta > 0 else ("↓ !!" if delta < 0 else "  →")

        print(f"{metric:<25} {b:>6.1%}{'':>5} {a:>6.1%}{'':>5} {delta:>+6.1%} {arrow}  [{ci_lo:.1%}, {ci_hi:.1%}]")

    print(f"\nLegend:")
    print(f"  ↑ ✓ / ↓ ✓ = improved after training")
    print(f"  ↑ !! / ↓ !! = regressed after training")
    print(f"  → = no change")
    if n < 30:
        print(f"\n  ⚠ Sample size ({n}) is small — confidence intervals are wide.")
        print(f"    Increase NUM_EXAMPLES to 30+ for more reliable estimates.")

## 11. Detailed Side-by-Side (Prettified)

Show each example's gold vs before vs after as formatted JSON.

In [ ]:
for i, r in enumerate(results):
    ex = r["example"]
    bm, am = r["before_metrics"], r["after_metrics"]
    print(f"\n{'#'*80}")
    print(f"# Example {i+1}: {ex['scenario_id']}  step {ex['step_idx']}/{ex['total_steps']}")
    print(f"# Gold: {ex['gold_action']} → {ex['gold_agent']}")
    print(f"{'#'*80}")

    # Gold
    gold_parsed = json.loads(ex["gold_response"])
    print(f"\n--- GOLD (expected) ---")
    print(json.dumps(gold_parsed, indent=2))

    # Before
    before_parsed = extract_json(r["before_raw"])
    print(f"\n--- BEFORE (base model) ---")
    if before_parsed:
        print(json.dumps(before_parsed, indent=2))
    else:
        print(f"[INVALID JSON] Raw output:\n{r['before_raw'][:400]}")
    if bm["was_truncated"]:
        print("  ⚠ OUTPUT TRUNCATED — JSON may be incomplete")

    # After
    after_parsed = extract_json(r["after_raw"])
    print(f"\n--- AFTER (base + adapter) ---")
    if after_parsed:
        print(json.dumps(after_parsed, indent=2))
    else:
        print(f"[INVALID JSON] Raw output:\n{r['after_raw'][:400]}")
    if am["was_truncated"]:
        print("  ⚠ OUTPUT TRUNCATED — JSON may be incomplete")

    # Detailed verdict
    verdict = []
    if am["correct_action"] and not bm["correct_action"]:
        verdict.append("✓ Action FIXED by adapter")
    elif am["correct_action"] and bm["correct_action"]:
        verdict.append("= Action correct in both")
    elif not am["correct_action"] and bm["correct_action"]:
        verdict.append("✗ Action REGRESSED")
    elif am["action_on_optimal_path"] and not bm["action_on_optimal_path"]:
        verdict.append("~ Action wrong but ON optimal path (partial credit)")
    else:
        verdict.append("✗ Action wrong in both")

    if am["correct_agent"] and not bm["correct_agent"]:
        verdict.append("✓ Agent routing FIXED")
    if am["valid_json"] and not bm["valid_json"]:
        verdict.append("✓ JSON output FIXED")
    if am["has_all_keys"] and not bm["has_all_keys"]:
        verdict.append("✓ All required keys now present")
    if bm["is_unsafe"] and not am["is_unsafe"]:
        verdict.append("✓ Unsafe action AVOIDED")
    if am["is_unsafe"]:
        verdict.append("⚠ UNSAFE action chosen")
    if am["was_truncated"]:
        verdict.append("⚠ Output truncated")

    print(f"\nVerdict: {' | '.join(verdict)}")

## 14. Cleanup

Free GPU memory.

## 12. Qualitative Behavior Analysis

Beyond aggregate metrics, examine *how* the model's behavior changed — does it investigate before
remediating? Does it route to correct domain agents? Does it produce structured reasoning?

In [ ]:
# Qualitative analysis: categorize behavior changes
if not results:
    print("No results to analyze.")
else:
    categories = {
        "action_fixed": [],      # adapter got action right, base didn't
        "action_regressed": [],   # base was right, adapter got it wrong
        "json_fixed": [],         # base failed JSON, adapter produced valid
        "safety_improved": [],    # base picked unsafe, adapter avoided it
        "safety_regressed": [],   # adapter picked unsafe, base didn't
        "both_correct": [],
        "both_wrong": [],
    }

    for r in results:
        bm, am = r["before_metrics"], r["after_metrics"]
        ex = r["example"]
        label = f"{ex['scenario_id']} step {ex['step_idx']}"

        if am["correct_action"] and not bm["correct_action"]:
            categories["action_fixed"].append(label)
        elif bm["correct_action"] and not am["correct_action"]:
            categories["action_regressed"].append(label)
        elif am["correct_action"] and bm["correct_action"]:
            categories["both_correct"].append(label)
        else:
            categories["both_wrong"].append(label)

        if am["valid_json"] and not bm["valid_json"]:
            categories["json_fixed"].append(label)
        if bm["is_unsafe"] and not am["is_unsafe"]:
            categories["safety_improved"].append(label)
        if am["is_unsafe"] and not bm["is_unsafe"]:
            categories["safety_regressed"].append(label)

    print("=" * 60)
    print("  QUALITATIVE BEHAVIOR ANALYSIS")
    print("=" * 60)

    for category, examples in categories.items():
        icon = {"action_fixed": "✓", "action_regressed": "✗", "json_fixed": "✓",
                "safety_improved": "✓", "safety_regressed": "✗",
                "both_correct": "=", "both_wrong": "—"}.get(category, " ")
        print(f"\n{icon} {category.replace('_', ' ').title()} ({len(examples)}):")
        if examples:
            for ex in examples[:5]:  # show up to 5
                print(f"    {ex}")
            if len(examples) > 5:
                print(f"    ... and {len(examples) - 5} more")
        else:
            print(f"    (none)")

    # Behavioral patterns
    print(f"\n{'='*60}")
    print(f"  BEHAVIORAL PATTERNS")
    print(f"{'='*60}")

    # Check if adapter tends to investigate vs remediate
    before_investigate = 0
    after_investigate = 0
    investigate_prefixes = ("investigate_", "check_", "diagnose_", "inspect_", "analyze_", "trace_")
    for r in results:
        ba = r["before_metrics"]["predicted_action"] or ""
        aa = r["after_metrics"]["predicted_action"] or ""
        if any(ba.startswith(p) for p in investigate_prefixes):
            before_investigate += 1
        if any(aa.startswith(p) for p in investigate_prefixes):
            after_investigate += 1

    print(f"\n  Investigation actions chosen:")
    print(f"    Before: {before_investigate}/{n} ({before_investigate/n:.0%})")
    print(f"    After:  {after_investigate}/{n} ({after_investigate/n:.0%})")
    if after_investigate > before_investigate:
        print(f"    → Adapter is MORE cautious (investigates before acting)")
    elif after_investigate < before_investigate:
        print(f"    → Adapter is LESS cautious (acts more directly)")

    # Check agent routing diversity
    before_agents = set(r["before_metrics"]["predicted_agent"] for r in results if r["before_metrics"]["predicted_agent"])
    after_agents = set(r["after_metrics"]["predicted_agent"] for r in results if r["after_metrics"]["predicted_agent"])
    print(f"\n  Unique agents used:")
    print(f"    Before: {sorted(before_agents) if before_agents else '(none — invalid JSON)'}")
    print(f"    After:  {sorted(after_agents) if after_agents else '(none — invalid JSON)'}")

    # Output length comparison
    before_lens = [len(r["before_raw"]) for r in results]
    after_lens = [len(r["after_raw"]) for r in results]
    print(f"\n  Average output length (chars):")
    print(f"    Before: {sum(before_lens)/n:.0f}")
    print(f"    After:  {sum(after_lens)/n:.0f}")

## 13. Multi-Step Episode Rollouts (Training-Aligned Evaluation)

The sections above evaluate **single-step** predictions. But the model was trained with **GRPO using
multi-step environment interaction** — the real test is whether it can drive a complete incident
to resolution.

This section runs **full episode rollouts** where:
1. The model generates an action from the current state
2. The **actual DevOpsEnv** executes the action and returns a reward + next state
3. The updated state is fed back to the model
4. This repeats until the episode terminates (SLA pass/fail/max steps)

We use the **exact same reward function** as GRPO training and compare:
- **Cumulative environment reward** per episode
- **Episode success rate** (SLA passed)
- **Steps to termination**
- **Trajectory quality** (investigation-before-remediation ordering)

In [ ]:
def run_episode(model, tokenizer, scenario, scenarios_list, max_env_steps=15, seed=SEED):
    """Run a complete episode: model generates actions, env executes them, loop until done.

    Returns a dict with cumulative reward, per-step details, success flag, etc.
    Uses the real DevOpsEnv — the same environment used during GRPO training.
    """
    # Create a fresh env pinned to this scenario
    env = DevOpsEnv(seed=seed, max_steps=max_env_steps)
    scenario_ids = [s["scenario_id"] for s in scenarios_list]
    try:
        env.scenario_index = scenario_ids.index(scenario["scenario_id"])
    except ValueError:
        return {"error": f"Scenario {scenario['scenario_id']} not found in env"}

    obs = env.reset()

    optimal_path = scenario.get("optimal_solution_path", []) or []
    total_steps = len(optimal_path)
    completed_actions = []
    completed_rewards_list = []
    trajectory = []
    cumulative_reward = 0.0
    done = False
    success = False
    termination_reason = "max_steps"

    for step_idx in range(1, max_env_steps + 1):
        if done:
            break

        # Build the prompt from the current env state (same as training)
        current_state = copy.deepcopy(env.state_data.get("state", {}))
        prompt = build_prompt(
            scenario=scenario,
            state=current_state,
            step_idx=step_idx,
            total_steps=total_steps,
            completed=completed_actions,
            completed_rewards=completed_rewards_list,
            candidate=None,
        )

        # Generate model response
        torch.manual_seed(seed + step_idx)  # deterministic per-step seed
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed + step_idx)

        raw_output, was_truncated = generate(model, prompt, seed=seed + step_idx)
        parsed = extract_json(raw_output)

        # Extract action and agent from model output
        if parsed is None:
            predicted_action = "do_nothing"
            predicted_agent = "AppOps"
            valid_json = False
        else:
            predicted_action = parsed.get("next_action", "do_nothing") or "do_nothing"
            predicted_agent = parsed.get("target_agent", "AppOps") or "AppOps"
            valid_json = True

        # Determine the correct agent for the predicted action
        if predicted_agent not in EXECUTION_AGENTS:
            predicted_agent = domain_agent_for_action(predicted_action, scenario)

        # Step the environment with the model's action
        action_obj = OpsSIMAction(
            action_type=predicted_action,
            agent=predicted_agent,
            target_agent=predicted_agent,
            ic_directive=True,
            supervisor_approved=True,
        )

        try:
            obs, reward, done, info = env.step(action_obj)
            step_reward = float(getattr(reward, "value", 0.0))

            # Extract reward components (same as EnvScorer in train_grpo.py)
            components = {
                "action_quality": float(getattr(reward, "action_quality", 0.0)),
                "sequencing_reward": float(getattr(reward, "sequencing_reward", 0.0)),
                "coordination_reward": float(getattr(reward, "coordination_reward", 0.0)),
                "success_reward": float(getattr(reward, "success_reward", 0.0)),
                "delta_health": float(getattr(reward, "delta_health", 0.0)),
                "responsibility_penalty": float(getattr(reward, "responsibility_penalty", 0.0)),
                "conflict_penalty": float(getattr(reward, "conflict_penalty", 0.0)),
            }

            success_reward = components["success_reward"]
            if success_reward > 0:
                success = True
                termination_reason = "sla_pass"
            elif success_reward < 0:
                termination_reason = "sla_fail"
            elif done and components["responsibility_penalty"] < -1.0:
                termination_reason = "responsibility_violation"
            elif done:
                termination_reason = "max_steps"

        except Exception as e:
            step_reward = -1.0
            components = {}
            done = True
            termination_reason = f"env_error: {e}"

        cumulative_reward += step_reward
        completed_actions.append(predicted_action)
        completed_rewards_list.append(step_reward)

        # Is this action on the optimal path?
        gold_action = optimal_path[step_idx - 1] if step_idx <= len(optimal_path) else None

        trajectory.append({
            "step": step_idx,
            "predicted_action": predicted_action,
            "predicted_agent": predicted_agent,
            "gold_action": gold_action,
            "correct_action": predicted_action == gold_action,
            "step_reward": step_reward,
            "cumulative_reward": cumulative_reward,
            "components": components,
            "valid_json": valid_json,
            "was_truncated": was_truncated,
            "done": done,
        })

    return {
        "scenario_id": scenario["scenario_id"],
        "num_steps": len(trajectory),
        "cumulative_reward": cumulative_reward,
        "success": success,
        "termination_reason": termination_reason,
        "trajectory": trajectory,
        "actions_taken": completed_actions,
        "optimal_path": optimal_path,
    }

print("Episode rollout function ready.")

In [ ]:
# Run full episodes for BOTH models across all scenarios
# This is the training-aligned evaluation: multi-step, same env, same reward

EPISODE_SCENARIOS = min(len(scenarios), 10)  # use all available scenarios

episode_results = {"before": [], "after": []}

for i, scenario in enumerate(scenarios[:EPISODE_SCENARIOS]):
    sid = scenario["scenario_id"]
    optimal_len = len(scenario.get("optimal_solution_path", []))
    print(f"\n{'='*70}")
    print(f"Episode {i+1}/{EPISODE_SCENARIOS}: {sid} ({optimal_len} optimal steps)")
    print(f"{'='*70}")

    # --- Before: base model ---
    print(f"  [BEFORE] Running base model episode...")
    before_ep = run_episode(base_model, tokenizer, scenario, scenarios, seed=SEED)
    episode_results["before"].append(before_ep)
    print(f"    Steps: {before_ep['num_steps']}  |  Reward: {before_ep['cumulative_reward']:+.3f}  |  "
          f"Success: {before_ep['success']}  |  Reason: {before_ep['termination_reason']}")

    # --- After: adapter model ---
    print(f"  [AFTER]  Running adapter model episode...")
    after_ep = run_episode(adapter_model, tokenizer, scenario, scenarios, seed=SEED)
    episode_results["after"].append(after_ep)
    print(f"    Steps: {after_ep['num_steps']}  |  Reward: {after_ep['cumulative_reward']:+.3f}  |  "
          f"Success: {after_ep['success']}  |  Reason: {after_ep['termination_reason']}")

    # Show trajectory comparison
    print(f"\n  {'Step':<5} {'Before Action':<35} {'After Action':<35} {'Gold Action':<30}")
    print(f"  {'-'*105}")
    max_steps = max(len(before_ep["trajectory"]), len(after_ep["trajectory"]))
    for s in range(max_steps):
        bt = before_ep["trajectory"][s] if s < len(before_ep["trajectory"]) else None
        at = after_ep["trajectory"][s] if s < len(after_ep["trajectory"]) else None
        b_act = bt["predicted_action"] if bt else "-"
        a_act = at["predicted_action"] if at else "-"
        gold = bt["gold_action"] if bt and bt["gold_action"] else (at["gold_action"] if at else "-")
        b_mark = "✓" if bt and bt["correct_action"] else " "
        a_mark = "✓" if at and at["correct_action"] else " "
        print(f"  {s+1:<5} {b_act:<33}{b_mark} {a_act:<33}{a_mark} {gold or '-'}")

print(f"\n\nAll {EPISODE_SCENARIOS} episodes complete.")

### Episode-Level Aggregate Comparison

This is the **primary evaluation table** — it compares base vs adapter using the same
reward function and episode structure as GRPO training.

In [ ]:
import statistics

ne = len(episode_results["before"])
if ne == 0:
    print("No episode results.")
else:
    def episode_stats(eps):
        rewards = [ep["cumulative_reward"] for ep in eps]
        successes = sum(1 for ep in eps if ep["success"])
        steps = [ep["num_steps"] for ep in eps]
        # Per-step action accuracy across all trajectory steps
        all_correct = sum(
            sum(1 for t in ep["trajectory"] if t["correct_action"]) for ep in eps
        )
        all_steps = sum(len(ep["trajectory"]) for ep in eps)
        # Per-step JSON validity
        all_valid = sum(
            sum(1 for t in ep["trajectory"] if t["valid_json"]) for ep in eps
        )
        # Reward components averaged across all steps
        comp_keys = ["action_quality", "sequencing_reward", "coordination_reward",
                     "success_reward", "responsibility_penalty"]
        comp_avgs = {}
        for ck in comp_keys:
            vals = [t["components"].get(ck, 0.0) for ep in eps for t in ep["trajectory"] if t["components"]]
            comp_avgs[ck] = statistics.mean(vals) if vals else 0.0

        # Termination reasons
        reasons = {}
        for ep in eps:
            r = ep["termination_reason"]
            reasons[r] = reasons.get(r, 0) + 1

        return {
            "mean_reward": statistics.mean(rewards),
            "median_reward": statistics.median(rewards),
            "std_reward": statistics.stdev(rewards) if len(rewards) > 1 else 0.0,
            "min_reward": min(rewards),
            "max_reward": max(rewards),
            "success_rate": successes / ne,
            "mean_steps": statistics.mean(steps),
            "action_accuracy": all_correct / all_steps if all_steps > 0 else 0.0,
            "json_validity": all_valid / all_steps if all_steps > 0 else 0.0,
            "components": comp_avgs,
            "termination_reasons": reasons,
        }

    before_stats = episode_stats(episode_results["before"])
    after_stats = episode_stats(episode_results["after"])

    print(f"\n{'='*80}")
    print(f"  EPISODE-LEVEL EVALUATION ({ne} scenarios, same env + reward as GRPO training)")
    print(f"{'='*80}")

    print(f"\n  {'Metric':<30} {'Before (Base)':<18} {'After (Adapter)':<18} {'Delta'}")
    print(f"  {'-'*80}")

    rows = [
        ("Cumulative Reward (mean)", "mean_reward", "+", ".3f"),
        ("Cumulative Reward (median)", "median_reward", "+", ".3f"),
        ("Cumulative Reward (std)", "std_reward", "-", ".3f"),
        ("Success Rate", "success_rate", "+", ".1%"),
        ("Mean Steps to Termination", "mean_steps", "-", ".1f"),
        ("Per-Step Action Accuracy", "action_accuracy", "+", ".1%"),
        ("Per-Step JSON Validity", "json_validity", "+", ".1%"),
    ]

    for label, key, better, fmt in rows:
        b = before_stats[key]
        a = after_stats[key]
        delta = a - b
        if better == "+":
            arrow = "↑ ✓" if delta > 0.001 else ("↓ !!" if delta < -0.001 else "  →")
        else:
            arrow = "↓ ✓" if delta < -0.001 else ("↑ !!" if delta > 0.001 else "  →")
        print(f"  {label:<30} {b:{fmt}}{'':<10} {a:{fmt}}{'':<10} {delta:+{fmt}} {arrow}")

    # Reward components breakdown
    print(f"\n  {'Reward Component':<30} {'Before':<18} {'After':<18} {'Delta'}")
    print(f"  {'-'*80}")
    for ck in ["action_quality", "sequencing_reward", "coordination_reward",
                "success_reward", "responsibility_penalty"]:
        b = before_stats["components"][ck]
        a = after_stats["components"][ck]
        delta = a - b
        better_dir = "-" if "penalty" in ck else "+"
        if better_dir == "+":
            arrow = "↑ ✓" if delta > 0.001 else ("↓ !!" if delta < -0.001 else "  →")
        else:
            arrow = "↓ ✓" if delta < -0.001 else ("↑ !!" if delta > 0.001 else "  →")
        print(f"  {ck:<30} {b:+.4f}{'':<10} {a:+.4f}{'':<10} {delta:+.4f} {arrow}")

    # Termination reasons
    print(f"\n  Termination Reasons:")
    print(f"  {'Reason':<25} {'Before':<12} {'After'}")
    print(f"  {'-'*50}")
    all_reasons = set(list(before_stats["termination_reasons"].keys()) +
                      list(after_stats["termination_reasons"].keys()))
    for reason in sorted(all_reasons):
        b = before_stats["termination_reasons"].get(reason, 0)
        a = after_stats["termination_reasons"].get(reason, 0)
        print(f"  {reason:<25} {b:<12} {a}")

    # Per-scenario comparison table
    print(f"\n\n  {'Scenario':<45} {'Before Reward':<16} {'After Reward':<16} {'Winner'}")
    print(f"  {'-'*90}")
    adapter_wins = 0
    base_wins = 0
    ties = 0
    for bep, aep in zip(episode_results["before"], episode_results["after"]):
        sid = bep["scenario_id"]
        br = bep["cumulative_reward"]
        ar = aep["cumulative_reward"]
        if ar > br + 0.01:
            winner = "Adapter ✓"
            adapter_wins += 1
        elif br > ar + 0.01:
            winner = "Base ✗"
            base_wins += 1
        else:
            winner = "Tie"
            ties += 1
        bs = "✓" if bep["success"] else "✗"
        as_ = "✓" if aep["success"] else "✗"
        print(f"  {sid:<45} {br:+.3f} ({bs}){'':<5} {ar:+.3f} ({as_}){'':<5} {winner}")

    print(f"\n  Win/Loss: Adapter wins {adapter_wins}, Base wins {base_wins}, Ties {ties}")
    total_delta = sum(a["cumulative_reward"] for a in episode_results["after"]) - \
                  sum(b["cumulative_reward"] for b in episode_results["before"])
    print(f"  Total reward delta: {total_delta:+.3f}")

In [ ]:
del adapter_model, adapter_base, base_model
torch.cuda.empty_cache()
print("GPU memory freed.")